# LSTM Deep Learning Model

## Business Objective

The objective of this notebook is to develop a Long Short-Term Memory (LSTM) neural network for forecasting burger sales using historical sales patterns, weather variables, and engineered time-based features.

Unlike traditional machine learning algorithms, LSTMs can learn temporal dependencies by processing sequential observations, making them well suited for time-series forecasting.

The model developed in this notebook will later be compared with the LightGBM model to determine which approach provides better forecasting performance.

In [316]:
# ==========================================================
# LSTM Deep Learning Model
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

plt.style.use("ggplot")

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


In [318]:
# ==========================================================
# Project Paths
# ==========================================================

from pathlib import Path

# Project root folder
PROJECT_ROOT = Path.cwd().parent

# Frequently used directories
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print("Project Root :", PROJECT_ROOT)
print("Data Folder  :", DATA_DIR)

Project Root : C:\Users
Data Folder  : C:\Users\data


In [319]:
# ==========================================================
# Load Prepared Sequences
# ==========================================================

X_train = np.load(DATA_DIR / "X_train_seq.npy")
X_validation = np.load(DATA_DIR / "X_validation_seq.npy")
X_test = np.load(DATA_DIR / "X_test_seq.npy")

y_train = np.load(DATA_DIR / "y_train_seq.npy")
y_validation = np.load(DATA_DIR / "y_validation_seq.npy")
y_test = np.load(DATA_DIR / "y_test_seq.npy")

print("Training:", X_train.shape)
print("Validation:", X_validation.shape)
print("Testing:", X_test.shape)

Training: (17065, 30, 19)
Validation: (3633, 30, 19)
Testing: (3634, 30, 19)


In [323]:
# ==========================================================
# Build LSTM Model
# ==========================================================

model = Sequential()

# ----------------------------------------------------------
# First LSTM Layer
# ----------------------------------------------------------
model.add(
    LSTM(
        units=64,
        return_sequences=True,
        input_shape=(30, 19)
    )
)

# Prevent overfitting
model.add(Dropout(0.20))

# ----------------------------------------------------------
# Second LSTM Layer
# ----------------------------------------------------------
model.add(
    LSTM(
        units=32
    )
)

# Prevent overfitting
model.add(Dropout(0.20))

# ----------------------------------------------------------
# Dense Hidden Layer
# ----------------------------------------------------------
model.add(
    Dense(
        units=16,
        activation="relu"
    )
)

# ----------------------------------------------------------
# Output Layer
# ----------------------------------------------------------
model.add(
    Dense(1)
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                        │ (None, 30, 64)              │          21,504 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 30, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 34,465 (134.63 KB)

 Trainable params: 34,465 (134.63 KB)

 Non-trainable params: 0 (0.00 B)

# Building the LSTM Architecture

The LSTM neural network is designed to learn temporal patterns from historical observations.

The architecture consists of:

- **First LSTM Layer (64 units):** Learns high-level temporal patterns from the previous 30 days.
- **Dropout Layer (20%):** Reduces overfitting by randomly disabling neurons during training.
- **Second LSTM Layer (32 units):** Learns more refined sequential relationships from the first LSTM layer.
- **Second Dropout Layer (20%):** Further improves the model's ability to generalize.
- **Dense Hidden Layer (16 neurons):** Combines the extracted features before making the final prediction.
- **Output Layer (1 neuron):** Predicts the burger sales for the next day.

The architecture is intentionally kept relatively simple because the dataset is moderate in size. A deeper network could increase the risk of overfitting without providing significant performance improvements.

In [321]:
# ==========================================================
# Compile Model
# ==========================================================

model.compile(

    optimizer="adam",

    loss="mse",

    metrics=["mae"]

)

print("Model Compiled Successfully!")

Model Compiled Successfully!


In [322]:
# ==========================================================
# Training Callbacks
# ==========================================================

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

early_stop = EarlyStopping(

    monitor="val_loss",

    patience=15,

    restore_best_weights=True
)

checkpoint = ModelCheckpoint(

    "../models/best_lstm.keras",

    monitor="val_loss",

    save_best_only=True
)

reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=5,

    min_lr=1e-6,

    verbose=1
)

# Model Training

The LSTM model is trained using the training dataset while monitoring its performance on the validation dataset.

Three callbacks are used during training:

- **EarlyStopping** prevents unnecessary training once validation performance stops improving.
- **ModelCheckpoint** saves the best-performing model during training.
- **ReduceLROnPlateau** automatically lowers the learning rate when the validation loss stops improving.

These techniques improve model generalization and reduce the likelihood of overfitting.

In [324]:
# ==========================================================
# Train the LSTM Model
# ==========================================================

# Ensure model is compiled before training (prevents ValueError)
if not hasattr(model, "optimizer") or model.optimizer is None:
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = model.fit(

    X_train,
    y_train,

    validation_data=(
        X_validation,
        y_validation
    ),

    epochs=100,

    batch_size=32,

    callbacks=[
        early_stop,
        checkpoint,
        reduce_lr
    ],

    verbose=1
)

print("Training Complete!")

ValueError: You must call `compile()` before using the model.